# A Practical Guide to Quantitative Finance Interviews — Chapter 6
## Finance / Option Pricing

Worked solutions to the option-pricing questions in **Xinfeng Zhou's Green Book**, in the style of the earlier
chapters: each concept is explained, then its interview questions are solved **step by step**, and every closed
form is cross-checked numerically. Concepts already built in Chapter 5 (Brownian motion, Itô's lemma) are
restated for reference where they are used.

### Notation used throughout Chapter 6
| symbol | meaning |
|---|---|
| $T$ | maturity date |
| $t$ | the current time |
| $\tau=T-t$ | time to maturity |
| $S$ | stock price at time $t$ |
| $r$ | continuous risk-free interest rate |
| $y$ | continuous dividend yield |
| $\sigma$ | annualized asset volatility |
| $c,\ p$ | price of a European **call** / **put** |
| $C,\ P$ | price of an American **call** / **put** |
| $D$ | present value, at $t$, of future dividends |
| $K$ | strike price |
| $PV$ | present value at $t$ |

(These are referenced by their full names now and then for ease of reading.)

## 6.1 Option Pricing

An **option** gives the right (not the obligation) to trade the stock at the strike $K$: a **call** to buy, a
**put** to sell. *European* options exercise only at maturity $T$; *American* options exercise any time up to
$T$. Everything below flows from **no-arbitrage** plus two Chapter-5 tools, restated here for reference:

- **Geometric Brownian motion (GBM)** for the stock: $dS=\mu S\,dt+\sigma S\,dW$, i.e.
  $S_T=S\,e^{(\mu-\sigma^2/2)\tau+\sigma(W_T-W_t)}$ — so $\ln S_T$ is normal (the stock is *log-normal*).
- **Itô's lemma:** for $f(t,S)$ with $dS=\mu S\,dt+\sigma S\,dW$,
$$df=\Big(f_t+\mu S f_S+\tfrac12\sigma^2S^2 f_{SS}\Big)dt+\sigma S f_S\,dW,$$
  and a process is a **martingale iff its $dt$ (drift) term is zero**.

The **risk-neutral** principle ties them together: under the risk-neutral measure the drift of the stock is the
risk-free rate ($\mu\to r$), and any European claim is worth the **discounted expected payoff**
$V=e^{-r\tau}\,E^{Q}[\text{payoff}(S_T)]$.

### 6.1.1 Price direction of options (1)

**Question.** How do vanilla European/American call and put prices move when each of $S,\ K,\ \tau,\ \sigma,\ r,\ D$
**increases** (holding the rest fixed)?

Reason variable by variable:
- **Stock price $S\uparrow$:** a call (right to *buy* at fixed $K$) gains; a put (right to *sell*) loses.
- **Strike $K\uparrow$:** buying becomes worse (call $\downarrow$), selling becomes better (put $\uparrow$).
- **Time to maturity $\tau\uparrow$:** an **American** option only gains from more time (more exercise
  opportunities), so both $C,P\uparrow$. A **European** option usually gains too, but not always — a deep
  in-the-money European *put* can *lose* value with more time (its payoff is capped at $K$, whose present value
  shrinks), so European $\tau$ is **ambiguous**.
- **Volatility $\sigma\uparrow$:** options are convex in $S$ (limited downside, open upside), so more dispersion
  helps **both** calls and puts ($\uparrow$).
- **Interest rate $r\uparrow$:** raising $r$ discounts the strike more. A call (pays $-K$ later) gains; a put
  (receives $+K$ later) loses.
- **Dividends $D\uparrow$:** dividends drop the stock on the ex-date, hurting calls and helping puts.

**Summary — effect of *increasing* each variable:**

| increase ↑ | European call $c$ | European put $p$ | American call $C$ | American put $P$ |
|---|:---:|:---:|:---:|:---:|
| $S$ (stock) | $+$ | $-$ | $+$ | $-$ |
| $K$ (strike) | $-$ | $+$ | $-$ | $+$ |
| $\tau$ (time to maturity) | ? | ? | $+$ | $+$ |
| $\sigma$ (volatility) | $+$ | $+$ | $+$ | $+$ |
| $r$ (rate) | $+$ | $-$ | $+$ | $-$ |
| $D$ (dividends) | $-$ | $+$ | $-$ | $+$ |

("$?$" marks the genuinely ambiguous European-time-value cases; American values are monotone in $\tau$ because
a longer-dated American option contains the shorter-dated one as an exercise choice.) The code confirms every
unambiguous sign by bumping each input in the Black–Scholes price.

In [1]:
import math

def N(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def bs(S, K, r, sigma, T, call=True, q=0.0):
    """Black-Scholes price with continuous dividend yield q (=y)."""
    d1 = (math.log(S / K) + (r - q + sigma ** 2 / 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if call:
        return S * math.exp(-q * T) * N(d1) - K * math.exp(-r * T) * N(d2)
    return K * math.exp(-r * T) * N(-d2) - S * math.exp(-q * T) * N(-d1)

base = dict(S=100, K=100, r=0.05, sigma=0.2, T=1.0, q=0.02)   # q = dividend yield D
sign = lambda x: "+" if x > 1e-9 else ("-" if x < -1e-9 else "0")
def price(call, **over):
    a = dict(base); a.update(over)
    return bs(a['S'], a['K'], a['r'], a['sigma'], a['T'], call=call, q=a['q'])

c0, p0 = price(True), price(False)
print("effect of increasing each variable on the Black-Scholes price (sign of the change):")
print(f"{'var':>6} | call | put")
for var, over in [('S', dict(S=101)), ('K', dict(K=101)), ('tau', dict(T=1.01)),
                  ('sigma', dict(sigma=0.21)), ('r', dict(r=0.06)), ('D(q)', dict(q=0.03))]:
    print(f"{var:>6} |  {sign(price(True, **over)-c0)}   |  {sign(price(False, **over)-p0)}")

effect of increasing each variable on the Black-Scholes price (sign of the change):
   var | call | put
     S |  +   |  -
     K |  -   |  +
   tau |  +   |  +
 sigma |  +   |  +
     r |  +   |  -
  D(q) |  -   |  +


### 6.1.2 Put-call parity (2)

**Question.** State and prove put–call parity for **European** options on a **non-dividend** stock.

**Statement.**
$$\boxed{\,c+K e^{-r\tau}=p+S\,}.$$

**Proof by two portfolios with identical maturity payoffs.** Build:
- **Portfolio A:** one European **call** ($c$) $+$ a **zero-coupon bond** with face value $K$ (present value
  $Ke^{-r\tau}$). Cost today: $c+Ke^{-r\tau}$.
- **Portfolio B:** one European **put** ($p$) $+$ one **share** of the stock ($S$) — a *protective put*. Cost
  today: $p+S$.

Their payoffs at maturity $T$:

| at $T$ | $S_T\le K$ | $S_T>K$ |
|---|:---:|:---:|
| **A:** call $+$ bond | $0+K=K$ | $(S_T-K)+K=S_T$ |
| **B:** put $+$ stock | $(K-S_T)+S_T=K$ | $0+S_T=S_T$ |

Both portfolios pay $\max(S_T,K)$ in *every* state. Two portfolios with identical payoffs in all states must
have the same price today (else short the dearer, buy the cheaper for a riskless profit), so
$c+Ke^{-r\tau}=p+S$. The code checks it against Black–Scholes.

In [2]:
# Put-call parity check: c + K e^{-r tau} should equal p + S  (non-dividend: q = 0)
S, K, r, sigma, T = 100, 95, 0.05, 0.2, 1.0
c = bs(S, K, r, sigma, T, call=True, q=0.0)
p = bs(S, K, r, sigma, T, call=False, q=0.0)
lhs = c + K * math.exp(-r * T)      # portfolio A: call + zero-coupon bond (face K)
rhs = p + S                          # portfolio B: put + stock (protective put)
print(f"c = {c:.4f},  p = {p:.4f}")
print(f"A: c + K e^-rT = {lhs:.6f}")
print(f"B: p + S       = {rhs:.6f}   -> equal, parity holds")

c = 13.3465,  p = 3.7133
A: c + K e^-rT = 103.713260
B: p + S       = 103.713260   -> equal, parity holds


### 6.1.3 American vs European options (3)

#### 3A — never exercise an American call early on a non-dividend stock

With no dividends the American and European call have the **same** price, because it is never optimal to
exercise the American call before maturity. Three ways to see it.

**Solution 1 — the lower bound beats the exercise value.** For a non-dividend European call,
$$c\ \ge\ S-Ke^{-r\tau}\ >\ S-K\qquad(\text{since } Ke^{-r\tau}<K\text{ for } r>0).$$
The call *alive* is worth more than $S-K$, which is exactly what you would collect by exercising. Exercising
throws away both the remaining time value and the interest you could earn on the strike, so you never do it —
$C=c$.

**Solution 2 — dominate the early-exercise cash flows.** Suppose at time $t\,(<T)$ the call is in the money and
you are tempted to exercise. Compare two strategies:

| | at time $t$ | at maturity $T$ |
|---|---|---|
| **1. Exercise now**, sell the share, invest the cash | net $S_t-K$, invested at $r$ | $(S_t-K)e^{r\tau}$ |
| **2. Keep the call**, short one share, invest $S_t$ | short $+S_t$, invest it | $S_t e^{r\tau}-\min(S_T,K)$ |

In strategy 2 you close the short at $T$ by buying the share in the market for $S_T$ **or** exercising the call
for $K$, whichever is cheaper — cost $\min(S_T,K)$. The difference is
$$\text{(2)}-\text{(1)}=Ke^{r\tau}-\min(S_T,K)\ \ge\ Ke^{r\tau}-K=K\big(e^{r\tau}-1\big)>0.$$
Keeping the call **always** beats exercising, by at least $K(e^{r\tau}-1)$. So never exercise early.

**Solution 3 — Jensen's inequality (convex payoff).** The call payoff $(S-K)^+$ is a **convex** function of
$S$. Under the risk-neutral measure $E^Q[S_T]=Se^{r\tau}$, so by Jensen,
$$c=e^{-r\tau}E^Q\big[(S_T-K)^+\big]\ \ge\ e^{-r\tau}\big(E^Q[S_T]-K\big)^+=e^{-r\tau}\big(Se^{r\tau}-K\big)^+=\big(S-Ke^{-r\tau}\big)^+\ >\ S-K.$$
Again the alive call exceeds the exercise value $S-K$. The convexity is the picture below: the chord of a
convex payoff lies below the curve, so averaging *then* taking the payoff (holding) beats taking the payoff of
the average (exercising now).
```
  payoff
    |                         /(S-K)^+
    |                        /
    |          hold: E[(S_T-K)^+]  •   (on the curve, above the chord)
    |                      /  ·
    |                     / ·   chord (Jensen gap)
    |____________________/·___________________  S
    0                    K        Se^{r tau}
```

(With dividends the story changes — a large enough dividend can make early exercise of an American call
optimal just before the ex-date; that is the one exception.)

#### 3B — is there an arbitrage between a call struck at 80 and a put struck at 90?

**Question.** A European **call** on a non-dividend stock with strike $80$ trades at $8$; a European **put** on
the same stock with strike $90$ trades at $9$. Is there an arbitrage?

**Set up a candidate arbitrage and check its payoff.** Try the portfolio that is long the (cheaper, lower-strike)
call and short the (dearer, higher-strike) put, delta-hedged with the stock and a bond:
$$\text{long call}(80),\quad \text{short put}(90),\quad \text{short 1 share},\quad \text{lend } 90e^{-r\tau}.$$
Its payoff at $T$ is
$$(S_T-80)^+-(90-S_T)^+-S_T+90=\begin{cases}0,& S_T<80,\\ S_T-80\in[0,10),& 80\le S_T<90,\\ 10,& S_T\ge90,\end{cases}$$
which is $\ge0$ always and $>0$ whenever $S_T>80$ (positive probability). So the **payoff leg is a valid
arbitrage payoff** — the only question is whether it costs nothing to set up. Its cost today is
$$8-9-S+90e^{-r\tau}=90e^{-r\tau}-S-1.$$
So this is a genuine arbitrage **iff** $S\ge 90e^{-r\tau}-1$. The mirror portfolio (long put(90), short call(80),
long stock, borrow $80e^{-r\tau}$) is an arbitrage when $S\le 80e^{-r\tau}-1$. In between —
$$80e^{-r\tau}-1\ <\ S\ <\ 90e^{-r\tau}-1$$
— **neither works, and the two prices are arbitrage-free.**

**So the honest answer: it depends on the current stock price.** The two option quotes *alone* do not force an
arbitrage. For example at $S=85,\ r=0$ they are perfectly consistent: put–call parity at strike $90$ gives the
implied call $c(90)=p(90)+S-90=9+85-90=4$, and $c(80)-c(90)=8-4=4$ lies in the required $[0,\,90-80]=[0,10]$, so
a valid price system (a risk-neutral distribution) exists. But at $S=95,\ r=0$ the first portfolio costs
$90-95-1=-6<0$ — a free $6$ today with a non-negative future payoff — a clear arbitrage. The code prints the
setup cost across stock prices and the no-arbitrage band.

In [3]:
import math

def arb_costs(S, r=0.0, T=1.0):
    """Setup cost of the two candidate arbitrage portfolios for call(80)=8, put(90)=9.
    Negative cost with a >=0 payoff = arbitrage."""
    high = 8 - 9 - S + 90 * math.exp(-r * T)      # long call80, short put90, short stock, lend 90 e^-rT
    low = 9 - 8 + S - 80 * math.exp(-r * T)       # mirror
    return high, low

def high_payoff(ST):
    return max(ST - 80, 0) - max(90 - ST, 0) - ST + 90

print("payoff of the long-call/short-put/short-stock/lend portfolio (>=0 always):")
print("  S_T = 50, 85, 120  ->", [high_payoff(s) for s in (50, 85, 120)])
print("\nsetup cost by current stock price (r = 0):  arbitrage when a cost is negative")
for S in (70, 79, 85, 89, 95):
    hi, lo = arb_costs(S)
    tag = "ARB" if (hi < -1e-9 or lo < -1e-9) else "no-arb"
    print(f"  S={S:>3}: high-cost {hi:+.1f}, low-cost {lo:+.1f}   -> {tag}")
print("\nno-arbitrage band (r=0): 79 < S < 89  (there the quotes are consistent, e.g. S=85)")

payoff of the long-call/short-put/short-stock/lend portfolio (>=0 always):
  S_T = 50, 85, 120  -> [0, 5, 10]

setup cost by current stock price (r = 0):  arbitrage when a cost is negative
  S= 70: high-cost +19.0, low-cost -9.0   -> ARB
  S= 79: high-cost +10.0, low-cost +0.0   -> no-arb
  S= 85: high-cost +4.0, low-cost +6.0   -> no-arb
  S= 89: high-cost +0.0, low-cost +10.0   -> no-arb
  S= 95: high-cost -6.0, low-cost +16.0   -> ARB

no-arbitrage band (r=0): 79 < S < 89  (there the quotes are consistent, e.g. S=85)


### 6.1.4 The Black–Scholes–Merton differential equation (4)

**Question.** Write down the BSM PDE and sketch its derivation.

**The equation.** For a derivative $V(S,t)$ on a non-dividend stock,
$$\boxed{\ \frac{\partial V}{\partial t}+\tfrac12\sigma^{2}S^{2}\frac{\partial^{2}V}{\partial S^{2}}+rS\frac{\partial V}{\partial S}-rV=0\ }.$$

**Derivation (GBM + Itô + a riskless hedge).**
1. **Model the stock** as GBM: $dS=\mu S\,dt+\sigma S\,dW$.
2. **Itô's lemma** on $V(S,t)$:
$$dV=\Big(V_t+\mu S V_S+\tfrac12\sigma^{2}S^{2}V_{SS}\Big)dt+\sigma S V_S\,dW.$$
3. **Form a hedged portfolio** that cancels the random $dW$: hold the derivative and short $\Delta=V_S$ shares,
$\Pi=V-V_S\,S$. Then
$$d\Pi=dV-V_S\,dS=\Big(V_t+\tfrac12\sigma^{2}S^{2}V_{SS}\Big)dt,$$
the $\mu S V_S\,dt$ and $\sigma S V_S\,dW$ terms **cancel** — $\Pi$ is instantaneously **riskless**.
4. **No arbitrage** forces a riskless portfolio to earn $r$: $d\Pi=r\Pi\,dt=r\big(V-V_S S\big)dt$. Equating the
   two expressions for $d\Pi$ and cancelling $dt$,
$$V_t+\tfrac12\sigma^{2}S^{2}V_{SS}=r\big(V-S V_S\big)\ \Longrightarrow\ V_t+\tfrac12\sigma^{2}S^{2}V_{SS}+rSV_S-rV=0.$$

Notice $\mu$ has **vanished**: the hedge removes the drift, which is why option prices do not depend on the
stock's expected return — the essence of **risk-neutral** pricing. Each option is this same PDE with a
different terminal condition (a call: $V(S,T)=(S-K)^+$; a put: $(K-S)^+$).

### 6.1.5 The Black–Scholes formula (5)

With continuous dividend yield $y$, the European call and put are
$$c=Se^{-y\tau}N(d_1)-Ke^{-r\tau}N(d_2),\qquad p=Ke^{-r\tau}N(-d_2)-Se^{-y\tau}N(-d_1),$$
$$d_1=\frac{\ln(Se^{-y\tau}/K)+(r+\sigma^2/2)\tau}{\sigma\sqrt\tau}=\frac{\ln(S/K)+(r-y+\sigma^2/2)\tau}{\sigma\sqrt\tau},\qquad
d_2=\frac{\ln(S/K)+(r-y-\sigma^2/2)\tau}{\sigma\sqrt\tau}=d_1-\sigma\sqrt\tau,$$
where $N(x)=\int_{-\infty}^{x}\tfrac{1}{\sqrt{2\pi}}e^{-y^2/2}\,dy$ is the standard normal CDF and
$N'(x)=\tfrac{1}{\sqrt{2\pi}}e^{-x^2/2}$ its density. (For a futures underlying $y=r$; for an FX rate $y=r_f$,
the foreign rate.) The derivations below take the non-dividend case $y=0$.

#### 5A — the six assumptions behind Black–Scholes
1. The stock follows a **geometric Brownian motion** with **constant** drift $\mu$ and volatility $\sigma$
   (so $S_T$ is log-normal).
2. **No arbitrage.**
3. **Frictionless, continuous trading:** securities are infinitely divisible, short selling is allowed, and you
   can rehedge continuously.
4. **No transaction costs or taxes.**
5. A **constant** risk-free rate $r$ at which you may borrow and lend freely.
6. **No dividends** over the life of the option (the base version; a yield $y$ is the standard extension).

#### 5B — deriving the call by risk-neutral expectation

Under the **risk-neutral measure** the stock drifts at $r$:
$$S_T=S\,e^{(r-\sigma^2/2)\tau+\sigma\sqrt\tau\,Z},\qquad Z\sim N(0,1),$$
and the call is the discounted expected payoff $c=e^{-r\tau}E^{Q}\big[(S_T-K)^+\big]$.

**Step 1 — when is the call in the money?** $S_T>K\iff Z>\dfrac{\ln(K/S)-(r-\sigma^2/2)\tau}{\sigma\sqrt\tau}=:-d_2$.
So the option pays off for $Z>-d_2$.

**Step 2 — split the expectation.**
$$e^{r\tau}c=E^{Q}\big[(S_T-K)^+\big]=\underbrace{E^{Q}\big[S_T\,\mathbf 1_{Z>-d_2}\big]}_{(\mathrm I)}-\underbrace{K\,E^{Q}\big[\mathbf 1_{Z>-d_2}\big]}_{(\mathrm{II})}.$$

**Step 3 — the easy term (II).** $E^{Q}[\mathbf 1_{Z>-d_2}]=P(Z>-d_2)=N(d_2)$, so $(\mathrm{II})=K\,N(d_2)$.

**Step 4 — the stock term (I), completing the square.** With $\varphi(z)=\tfrac1{\sqrt{2\pi}}e^{-z^2/2}$,
$$(\mathrm I)=\int_{-d_2}^{\infty}S\,e^{(r-\sigma^2/2)\tau+\sigma\sqrt\tau\,z}\,\varphi(z)\,dz
=Se^{r\tau}\int_{-d_2}^{\infty}\tfrac1{\sqrt{2\pi}}e^{-\frac12(z-\sigma\sqrt\tau)^2}\,dz,$$
because $(r-\sigma^2/2)\tau+\sigma\sqrt\tau z-\tfrac{z^2}{2}=r\tau-\tfrac12(z-\sigma\sqrt\tau)^2$. Substituting
$w=z-\sigma\sqrt\tau$ turns the integral into $P(w>-d_2-\sigma\sqrt\tau)=N(d_2+\sigma\sqrt\tau)=N(d_1)$, so
$(\mathrm I)=Se^{r\tau}N(d_1)$.

**Step 5 — assemble and discount.**
$$e^{r\tau}c=Se^{r\tau}N(d_1)-K\,N(d_2)\ \Longrightarrow\ \boxed{\,c=S\,N(d_1)-Ke^{-r\tau}N(d_2)\,}.$$
The put follows from put–call parity. The code prices both and re-checks parity.

In [4]:
import math, random

def bs_call(S, K, r, sigma, T):
    d1 = (math.log(S / K) + (r + sigma ** 2 / 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * N(d1) - K * math.exp(-r * T) * N(d2)

def call_montecarlo(S, K, r, sigma, T, trials=1000000, seed=0):
    """c = e^{-rT} E^Q[(S_T - K)^+] under risk-neutral GBM."""
    rng = random.Random(seed); tot = 0.0
    for _ in range(trials):
        ST = S * math.exp((r - sigma ** 2 / 2) * T + sigma * math.sqrt(T) * rng.gauss(0, 1))
        tot += max(ST - K, 0.0)
    return math.exp(-r * T) * tot / trials

S, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0
print(f"call (formula)      = {bs_call(S,K,r,sigma,T):.4f}")
print(f"call (risk-neutral MC) = {call_montecarlo(S,K,r,sigma,T):.4f}")
p = bs(S, K, r, sigma, T, call=False, q=0.0)
print(f"put via parity check: c - p = {bs_call(S,K,r,sigma,T)-p:+.4f}  vs  S - K e^-rT = {S-K*math.exp(-r*T):+.4f}")

call (formula)      = 10.4506
call (risk-neutral MC) = 10.4645
put via parity check: c - p = +4.8771  vs  S - K e^-rT = +4.8771


#### 5C — deriving the call by solving the BSM PDE (reduction to the heat equation)

Solve $V_t+\tfrac12\sigma^2S^2V_{SS}+rSV_S-rV=0$ with $V(S,T)=(S-K)^+$ by turning it into the **heat
equation** $u_\theta=u_{xx}$, whose solution is known.

**Step 1 — log-price and reversed, rescaled time.** Put
$$x=\ln\frac{S}{K},\qquad \theta=\tfrac12\sigma^2(T-t),\qquad V(S,t)=K\,v(x,\theta).$$
Then $\partial_t=-\tfrac12\sigma^2\partial_\theta$, $S\partial_S=\partial_x$, $S^2\partial_{SS}=\partial_{xx}-\partial_x$, and
dividing by $\tfrac12\sigma^2 K$ the PDE becomes, with $k=\dfrac{r}{\tfrac12\sigma^2}=\dfrac{2r}{\sigma^2}$,
$$v_\theta=v_{xx}+(k-1)v_x-k\,v.$$

**Step 2 — kill the lower-order terms.** Look for $v=e^{\alpha x+\beta\theta}u(x,\theta)$. Choosing
$$\alpha=-\tfrac{k-1}{2},\qquad \beta=-\tfrac{(k+1)^2}{4}$$
cancels the $v_x$ and $v$ terms and leaves the pure **heat equation**
$$u_\theta=u_{xx}.$$

**Step 3 — the initial condition.** At $\theta=0$ (i.e. $t=T$), $V=(S-K)^+=K(e^x-1)^+$, so $v(x,0)=(e^x-1)^+$
and hence
$$u(x,0)=e^{-\alpha x}(e^x-1)^+=\big(e^{\frac{k+1}{2}x}-e^{\frac{k-1}{2}x}\big)^+ .$$

**Step 4 — solve the heat equation** with its Gaussian kernel,
$$u(x,\theta)=\frac{1}{\sqrt{4\pi\theta}}\int_{-\infty}^{\infty}u(s,0)\,e^{-(x-s)^2/4\theta}\,ds.$$
Each of the two exponential pieces is a Gaussian integral that **completes the square** into a standard-normal
tail — exactly the two integrals of Step 4 in 5B — producing two $N(\cdot)$ terms.

**Step 5 — unwind the substitutions.** Reversing $u\to v\to V$ and rewriting $x,\theta,k$ back in terms of
$S,\tau,r,\sigma$, the two normal terms become $N(d_1)$ and $N(d_2)$ with the $d_1,d_2$ above, giving
$$\boxed{\,c=S\,N(d_1)-Ke^{-r\tau}N(d_2)\,}$$
— the same formula as the probabilistic route, now as the unique solution of the PDE with the call's boundary
condition.

#### 5D — an option that pays 1 the first time the stock hits $H$

**Question.** Interest rate is $0$; a non-dividend stock is at $1$ today. The first time the price reaches a
level $H>1$ you may exercise and receive $1$. What is the option worth today?

**Brute force (risk-neutral).** With $r=0$ and no dividends, the discounted price is a **martingale** and there
is no discounting, so the option is worth $P(\text{the stock ever reaches }H)\times 1$. The stock $S_t$ is a
positive martingale started at $1$; run it until it either hits $H$ or drifts arbitrarily low. Optional
stopping on the martingale $S$ gives
$$E[S_\tau]=S_0=1\ \Longrightarrow\ P(\text{hit }H)\cdot H+P(\text{never})\cdot 0=1\ \Longrightarrow\ P(\text{hit }H)=\frac1H.$$
So the option is worth $\dfrac1H$.

**No-arbitrage replication.** Buy $\tfrac1H$ shares today, at cost $\tfrac1H\cdot 1=\tfrac1H$. If the stock ever
reaches $H$, those shares are worth $\tfrac1H\cdot H=1$ — exactly the payoff, which you deliver by selling them.
If the stock never reaches $H$ (it drifts to $0$), both the shares and the option end at $0$. The stock position
reproduces the option's payoff in every scenario, so by no-arbitrage the option is worth
$$\boxed{\dfrac1H}.$$

In [5]:
import math, random

def one_touch_value(H):
    """Zero-rate, non-dividend: value of receiving 1 when S first hits H, starting at S0=1.  = 1/H."""
    return 1.0 / H

def one_touch_sim(H, sigma=0.5, dt=0.05, trials=8000, cap=150.0, floor=0.02, seed=0):
    """P(martingale stock started at 1 reaches H before drifting to ~0)."""
    rng = random.Random(seed); hit = 0
    for _ in range(trials):
        s = 1.0; t = 0.0
        while t < cap and floor < s < H:
            s *= math.exp(-0.5 * sigma * sigma * dt + sigma * math.sqrt(dt) * rng.gauss(0, 1)); t += dt
        hit += (s >= H)
    return hit / trials

for H in (2.0, 3.0, 4.0):
    print(f"H={H}: value 1/H = {one_touch_value(H):.4f}   (Monte-Carlo P(hit H) ~ {one_touch_sim(H):.3f})")

H=2.0: value 1/H = 0.5000   (Monte-Carlo P(hit H) ~ 0.470)
H=3.0: value 1/H = 0.3333   (Monte-Carlo P(hit H) ~ 0.315)
H=4.0: value 1/H = 0.2500   (Monte-Carlo P(hit H) ~ 0.239)


#### 5E — a contract that pays $1/S_T$ at maturity

**Question.** A non-dividend stock follows geometric Brownian motion. What is the value today of a contract
paying $1/S_T$ at maturity $T$?

**Solution — Itô under the risk-neutral measure.** Risk-neutrally $dS=rS\,dt+\sigma S\,dW$. Apply Itô's lemma to
$f(S)=1/S$ (here $f_S=-1/S^2$, $f_{SS}=2/S^3$):
$$d\!\left(\frac1S\right)=-\frac{1}{S^2}\,dS+\tfrac12\frac{2}{S^3}(dS)^2=\frac1S\Big[(\sigma^2-r)\,dt-\sigma\,dW\Big].$$
So $1/S$ is itself a geometric Brownian motion with drift $\sigma^2-r$, giving
$$E^{Q}\!\Big[\frac{1}{S_T}\Big]=\frac{1}{S}\,e^{(\sigma^2-r)\tau}.$$
Discounting at $r$,
$$V=e^{-r\tau}E^{Q}\!\Big[\frac{1}{S_T}\Big]=\boxed{\ \frac{1}{S}\,e^{(\sigma^2-2r)\tau}\ }.$$
(Equivalently, write $1/S_T=\tfrac1S e^{-(r-\sigma^2/2)\tau-\sigma\sqrt\tau Z}$ and use
$E[e^{-\sigma\sqrt\tau Z}]=e^{\sigma^2\tau/2}$ — the same answer.) The extra $e^{\sigma^2\tau}$ over naive
$\tfrac1S e^{-r\tau}$ is the convexity of $1/x$: by Jensen $E[1/S_T]>1/E[S_T]$.

In [6]:
import math, random

def inverse_contract_value(S, r, sigma, T):
    """Value of a claim paying 1/S_T:  (1/S) e^{(sigma^2 - 2r) T}."""
    return (1.0 / S) * math.exp((sigma ** 2 - 2 * r) * T)

def inverse_contract_sim(S, r, sigma, T, trials=800000, seed=0):
    rng = random.Random(seed); tot = 0.0
    for _ in range(trials):
        ST = S * math.exp((r - sigma ** 2 / 2) * T + sigma * math.sqrt(T) * rng.gauss(0, 1))
        tot += 1.0 / ST
    return math.exp(-r * T) * tot / trials

S, r, sigma, T = 100, 0.05, 0.3, 1.0
print(f"value of 1/S_T contract: formula {inverse_contract_value(S,r,sigma,T):.6f}   "
      f"(risk-neutral MC {inverse_contract_sim(S,r,sigma,T):.6f})")
print(f"naive (ignoring convexity) 1/S e^-rT = {(1/S)*math.exp(-r*T):.6f}  <  correct value (Jensen)")

value of 1/S_T contract: formula 0.009900   (risk-neutral MC 0.009896)
naive (ignoring convexity) 1/S e^-rT = 0.009512  <  correct value (Jensen)


---
*More of Chapter 6 as I keep reading.*